# Profilování regionální zátěže sítě a výpadků pomocí PROC CHART

## Shrnutí pro vedení

Analytik provozu sítě používá PROC CHART k profilování syntetického vzorku měření na vývodových okruzích napříč čtyřmi obslužnými regiony a čtyřmi zdroji výroby. Notebook prochází svislé sloupcové, vodorovné sloupcové, blokové a koláčové grafy, aby shrnul skladbu výroby, porovnal průměrnou a celkovou zátěž podle regionu, odhalil večerní špičku poptávky podle hodiny a seřadil minuty výpadků podle zdroje — jde o rychlý, textově orientovaný průzkum, jaký tým z energetiky a utilit spouští před tím, než se zaváže ke grafickému reportu. DATA step požaduje 1 200 řádků; tento notebook byl vykreslen v nelicencovaném režimu, který omezuje pracovní datovou sadu na prvních 100 měření, takže každý graf níže shrnuje tento vzorek o 100 řádcích.

## Datové zdroje

| Datová sada | Řádky | Popis |
|---------|------|-------------|
| `grid_ops` | 100 (syntetický vzorek) | Jeden řádek na měření vývodového okruhu v regionální síti, generovaný inline pomocí `call streaminit(20260531)` a `rand()`. Smyčka DATA step si vyžádá 1 200 měření, ale nelicencovaný režim omezuje uloženou datovou sadu na 100 pozorování, takže grafy níže popisují těchto 100 řádků. |

# Profilování regionální zátěže sítě a výpadků pomocí PROC CHART

PROC CHART vytváří znakové sloupcové, blokové a koláčové grafy přímo ve výpisu — bez potřeby zařízení ODS Graphics. Díky tomu je to rychlý nástroj pro první průzkum pro tým provozu sítě, který chce *vidět* tvar svých dat o zátěži a spolehlivosti dříve, než začne budovat vyladěné vizuály GCHART nebo SGPLOT.

V tomto notebooku:

1. Vygenerujeme syntetický měsíc měření vývodových okruhů pro provozovatele regionální sítě.
2. Zobrazíme graf **skladby výroby** (podíl měření podle zdroje).
3. Porovnáme **průměrnou a celkovou dodanou zátěž** napříč obslužnými regiony.
4. Odhalíme **večerní špičku poptávky** podle hodiny dne.
5. Použijeme **blokový graf** k porovnání regionu se zdrojem výroby.
6. Seřadíme **minuty výpadků** podle zdroje a najdeme nejméně spolehlivý přívod.

Každý příkaz a volba níže představuje standardní syntaxi PROC CHART v SAS 9.4.

## Krok 1 — Vygenerování syntetických provozních dat

DATA step níže vytváří měření v smyčce o 1 200 iteracích. Každému měření je přiřazen obslužný region, zdroj výroby (dominuje Gas, zbytek tvoří Wind, Solar a Hydro) a hodina dne. Zátěž je vyšší ve večerním okně 17:00–21:00 (a tato měření označujeme jako špičková) a minuty výpadků čerpáme z Poissonova rozdělení. `call streaminit` fixuje náhodné semínko, takže data jsou reprodukovatelná.

NOTE v logu ukazuje praktický výsledek: tento běh je v nelicencovaném režimu, takže uložená datová sada `grid_ops` je omezena na prvních 100 z těchto měření (100 řádků, 7 sloupců). Každý následující graf shrnuje tento vzorek o 100 řádcích a každý log PROC CHART potvrzuje, že přečetl 100 řádků.

In [1]:
/* Synthetic feeder-circuit operations for a regional grid operator */
data grid_ops;
    CALL streaminit(20260531);
    DÉLKA region $12 source $9 peak_flag $3;
    POLE regions[4] $12 _temporary_
        ('North','South','East','West');
    OPAKUJ meter_id = 1 TO 1200;
        r = ceil(4 * rand('uniform'));
        region = regions[r];
        u = rand('uniform');
        KDYŽ u < 0.42 PAK source = 'Gas';
        JINAK KDYŽ u < 0.70 PAK source = 'Wind';
        JINAK KDYŽ u < 0.88 PAK source = 'Solar';
        JINAK source = 'Hydro';
        hour = floor(24 * rand('uniform'));
        BASE = 18 + 5 * (region = 'North') + 3 * (region = 'West');
        KDYŽ hour >= 17 AND hour <= 21 PAK OPAKUJ;
            load_mwh = BASE + 12 + 6 * rand('normal');
            peak_flag = 'Yes';
        KONEC;
        JINAK OPAKUJ;
            load_mwh = BASE + 4 * rand('normal');
            peak_flag = 'No';
        KONEC;
        KDYŽ load_mwh < 0 PAK load_mwh = 0;
        outage_min = rand('poisson', 2.5);
        VÝSTUP;
    KONEC;
    ODSTRANIT r u BASE;
SPUSTIT;

NOTE: DATA grid_ops

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote grid_ops (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds


## Krok 2 — Skladba výroby

Jaký podíl měření připadá na každý zdroj výroby? Svislý sloupcový graf s `TYPE=PERCENT` odpovídá přímo: výšky sloupců představují procentní podíl všech pozorování spadajících do každé kategorie zdroje. Protože `source` je znaková proměnná, PROC CHART automaticky považuje její hodnoty za diskrétní kategorie.

In [2]:
PROCEDURA chart data=grid_ops;
    VBAR source / type=PERCENT;
SPUSTIT;


Percent of SOURCE

         |              *****       
         |              *****       
   40.00 +              *****       
         |              *****       
         |              *****       
   30.00 +              *****       
         |       *****  *****       
         |       *****  *****       
   20.00 +       *****  *****       
         |       *****  *****       
         |*****  *****  *****       
         |*****  *****  *****  *****
   10.00 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |                          
         +--------------------------+
          Hydro  Wind    Gas   Solar
                    SOURCE


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Krok 3 — Průměrná dodaná zátěž podle regionu

Nyní přecházíme od počítání k shrnutí měřené veličiny. Uvedení `load_mwh` v `SUMVAR=` s `TYPE=MEAN` způsobí, že délka sloupce představuje průměrnou zátěž na region, a explicitně si vyžádáme sloupce se statistikami: `MEAN` vytiskne průměr vedle každého sloupce a `CFREQ` přidá sloupec s kumulativní četností. North má nejvyšší průměrnou dodanou zátěž (průměr přibližně 28,0 MWh), což odpovídá regionálnímu posunu zabudovanému do dat, zatímco South je nejnižší (přibližně 19,8 MWh).

Také předáváme `ASCENDING` s úmyslem seřadit sloupce od nejnižšího po nejvyšší průměr. Všimněte si ve výstupu, že sloupce **nejsou** přeuspořádány — objevují se v pořadí kategorií (West, South, East, North), s průměry 24,2, 19,8, 21,7, 28,0, které nejdou od nejkratšího po nejdelší. Přeuspořádání sloupců podle statistiky grafu zatím není v této implementaci PROC CHART zapojeno, takže `ASCENDING`/`DESCENDING` jsou přijaty, ale aktuálně nemají žádný efekt; pořadí čtěte místo toho z vytištěného sloupce `Mean`.

In [3]:
PROCEDURA chart data=grid_ops;
    HBAR region / SUMVAR=load_mwh type=mean
                  cfreq mean ascending;
SPUSTIT;


Mean of REGION

REGION                                                  Mean           N     Percent
                                                                                    
  West  **********************************             24.17       24.17       23.00
 South  ****************************                   19.80       43.97       21.00
  East  *******************************                21.72       65.69       29.00
 North  ****************************************       28.03       93.72       27.00


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Krok 4 — Celková zátěž podle regionu

Zde `TYPE=SUM` způsobí, že každý sloupec představuje *celkovou* dodanou zátěž regionu spíše než průměr, takže vyšší sloupce označují regiony dodávající nejvíce agregované energie napříč vzorkovanými měřeními. North opět vede v celkové dodané zátěži.

Příkaz také požaduje `SUBGROUP=peak_flag`, čímž žádá PROC CHART, aby rozdělil každý sloupec podle toho, zda měření spadalo do večerního špičkového okna. V SASu to segmentuje každý sloupec odlišným glyfem pro každou úroveň podskupiny a vytiskne legendu. Tato implementace zatím nevykresluje segmentaci podskupin (zdokumentovaná mezera ve schopnostech), takže sloupce vycházejí jako plné běhy `*` bez rozdělení na jednotlivé segmenty — *součty* sloupců jsou správné, ale rozdělení špička vs. mimo špičku zde není zobrazeno. Chcete-li vidět, kolik zátěže spadá do špičkových hodin, použijte pohled podle hodiny dne v kroku 5.

In [4]:
PROCEDURA chart data=grid_ops;
    VBAR region / SUMVAR=load_mwh type=sum
                  SUBGROUP=peak_flag;
SPUSTIT;


Sum of REGION

         |                     *****
         |                     *****
         |                     *****
     600 +              *****  *****
         |*****         *****  *****
         |*****         *****  *****
         |*****         *****  *****
     400 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
     200 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |                          
         +--------------------------+
          West   South  East   North
                    REGION


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Krok 5 — Tvar zátěže během dne

`hour` je spojitá, takže binování si sami pevně nastavíme explicitním seznamem `MIDPOINTS=` ve 4hodinových středech (2, 6, 10, 14, 18, 22). Každý sloupec zobrazuje průměrnou dodanou zátěž pro měření poblíž dané hodiny. Sloupec vystředěný na 18 by měl vyčnívat — to je večerní špička poptávky, kterou jsme do dat zabudovali.

In [5]:
PROCEDURA chart data=grid_ops;
    VBAR hour / SUMVAR=load_mwh type=mean
                midpoints=2 6 10 14 18 22;
SPUSTIT;


Mean of HOUR

         |                            *****       
         |                            *****       
   30.00 +                            *****       
         |                            *****  *****
         |                            *****  *****
         |              *****         *****  *****
   20.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
   10.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |                                        
         +----------------------------------------+
            2      6     10     14     18     22  
                            HOUR


NOTE: PROC CHART data=grid_ops


## Krok 6 — Region podle zdroje výroby (blokový graf)

Blokový graf (BLOCK) vykresluje malou dvourozměrnou tabulku jako mřížku bloků. Pomocí `GROUP=source` a `SUMVAR=load_mwh / TYPE=MEAN` graf porovnává každý region se zdrojem výroby, který jej obsluhuje, přičemž výška bloku je úměrná průměrné zátěži — kompaktní způsob, jak odhalit, které kombinace region/zdroj nesou nejvyšší průměrnou zátěž.

In [6]:
PROCEDURA chart data=grid_ops;
    BLOCK region / SUMVAR=load_mwh type=mean
                   GROUP=source;
SPUSTIT;


Block chart of Mean of REGION by SOURCE

                          /####\
  /####\                  /####\
  /####\          /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  +----+  +----+  +----+  +----+
   West   South    East   North 
             REGION


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Krok 7 — Skladba výroby jako koláčový graf

Tatáž informace o podílu zdrojů z kroku 2, vykreslená jako koláč. PIE s `TYPE=PERCENT` velikost každého výseče určuje podle jeho procentního podílu na celkových měřeních a vedle obrázku vytiskne legendu s procenty výsečí.

In [7]:
PROCEDURA chart data=grid_ops;
    PIE source / type=PERCENT;
SPUSTIT;


Pie chart of SOURCE

          SOURCE     Percent   Percent  Slice
----------------  ----------  --------  ------------------------------
           Hydro       14.00     14.0%  ####
            Wind       28.00     28.0%  ********
             Gas       45.00     45.0%  ++++++++++++++
           Solar       13.00     13.0%  OOOO


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Krok 8 — Minuty výpadků podle zdroje

Nakonec seřadíme spolehlivost. `SUMVAR=outage_min` s `TYPE=SUM` sečte minuty výpadků na zdroj. Předáváme `DESCENDING` ve snaze vynést nejhůře fungující zdroj nahoru, ale stejně jako v kroku 3 nejsou sloupce přeuspořádány — tisknou se v pořadí kategorií (Hydro, Wind, Gas, Solar) a přeuspořádání sloupců zatím není implementováno. Pořadí čtěte z vytištěného sloupce `Sum`: Gas představuje v tomto vzorku nejvíce celkových minut výpadků (122), výrazně před Wind (64), Solar (43) a Hydro (38). To odpovídá skladbě výroby — Gas obsluhuje nejvíce měření, takže celkově nasbírá nejvíce minut výpadků.

In [8]:
PROCEDURA chart data=grid_ops;
    HBAR source / SUMVAR=outage_min type=sum SESTUPNĚ;
SPUSTIT;


Sum of SOURCE

SOURCE                                                   Sum        Cum.     Percent
                                                                     Sum            
 Hydro  ************                                      38          38       14.00
  Wind  *********************                             64         102       28.00
   Gas  ****************************************         122         224       45.00
 Solar  **************                                    43         267       13.00


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Interpretace výsledků

Čtení grafů dohromady dává provoznímu týmu rychlý situační přehled (nad vzorkem o 100 řádcích, který tento běh zachytil):

- **Skladba výroby (kroky 2 a 7).** Gas nese největší podíl měření (přibližně 45 %), Wind je druhý (přibližně 28 %) a Hydro a Solar zaostávají (přibližně 14 % a 13 %) — svislý sloupcový a koláčový graf vyprávějí stejný příběh dvěma způsoby, což je užitečná kontrola správnosti.
- **Zátěž podle regionu (kroky 3 a 4).** HBAR s průměrnou zátěží ukazuje North s nejvyšší průměrnou dodanou zátěží (průměr ~28 MWh) a South s nejnižší (~20 MWh), v souladu s regionálním posunem zabudovaným do dat. Graf SUM potvrzuje, že North dodává také nejvíce celkové energie.
- **Denní tvar zátěže (krok 5).** Sloupec středu vystředěný na hodinu 18 je jednoznačně nejvyšší, což potvrzuje špičku poptávky 17:00–21:00, kterou jsme do dat zabudovali — přesně tam, kam by utilita zaměřila řízení poptávky a plánování kapacity.
- **Spolehlivost (krok 8).** Sečtení minut výpadků podle zdroje odhaluje Gas jako největšího přispěvatele k výpadkům v tomto vzorku (122 minut), přirozený další cíl pro revizi údržby — i když to většinou odráží fakt, že Gas obsluhuje nejvíce měření.

Dvě zde použité volby zobrazení — přeuspořádání sloupců `ASCENDING`/`DESCENDING` (kroky 3 a 8) a segmentace sloupců `SUBGROUP=` (krok 4) — jsou přijaty parserem, ale zatím nejsou touto implementací PROC CHART vykreslovány, takže pořadí a rozdělení se čtou z vytištěných statistických sloupců spíše než z pořadí či stínování sloupců.

PROC CHART poskytuje pouze znakový výstup, takže pro vizuály připravené pro vedení by tým tyto stejné pohledy přenesl do PROC GCHART nebo PROC SGPLOT. Ale jako první průchod bez jakéhokoli nastavení nad čerstvým výběrem tyto grafy odpovídají na provozní otázky — skladbu, velikost a načasování — během několika sekund.